# Leaf Disease Detection — Colab runner

This notebook is only a launcher: every step calls a script in `src/`.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

**How to use**
1. Run cells 1–4 once per session (GPU check, get the code + install, Drive, data).
2. Run the experiment cells in order. Each one **skips itself if its results already exist in Drive**, and the
   classifier runs use `--resume`, so after a disconnect just re-run cells 1–4 and continue.
3. Run the last two cells to build the report and download it.

Results live in Google Drive (`MyDrive/apple_leaf_runs`), so they survive disconnects. Data lives on the
Colab disk (`/content/data`) and must be re-prepared in each new session (cell 4).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code from GitHub and install
Clones the repo on the first run of a session, or pulls the latest changes if it's already there.

In [ ]:
import os
REPO_URL = 'https://github.com/tajammulbasheer/plant_leaf_disease_detection.git'
REPO_DIR = '/content/plant_leaf_disease_detection'
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format='Code version: %h %s'
!pip install -q -r requirements.txt

## 3. Mount Google Drive (results are saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/apple_leaf_runs'
DATA = '/content/data'
os.makedirs(RUN_DIR, exist_ok=True)

def done(path):
    """True if a step's final output already exists in Drive."""
    exists = os.path.exists(path)
    if exists:
        print(f'Already done, skipping: {path}')
    return exists

## 4. Prepare the data (each new session, ~10–15 min)
Clones the original PlantVillage repo, makes a seeded 80/20 split of all 38 classes, an Apple-only subset, and U-Net masks.

In [ ]:
if not os.path.exists(f'{DATA}/plantvillage_full/train'):
    !python -m src.data.prepare_plantvillage --output-dir {DATA} --cleanup-clone
for sub in ['plantvillage_full/train', 'apple/train', 'seg/images']:
    p = f'{DATA}/{sub}'
    print(sub, '->', len(os.listdir(p)) if os.path.exists(p) else 'MISSING')

## 5. Classifier on all 38 classes
Three modes. `--resume` continues an interrupted run from its last saved epoch.

In [ ]:
!python -m src.training.train_classifier --data-dir {DATA}/plantvillage_full \
    --output-dir {RUN_DIR}/baseline_full --mode full --epochs 20 --resume

In [ ]:
!python -m src.training.train_classifier --data-dir {DATA}/plantvillage_full \
    --output-dir {RUN_DIR}/few_shot_k20 --mode few_shot --k-shot 20 --epochs 25 --resume

In [ ]:
!python -m src.training.train_classifier --data-dir {DATA}/plantvillage_full \
    --output-dir {RUN_DIR}/semi_supervised_k20 --mode semi_supervised --k-shot 20 --epochs 25 \
    --confidence-threshold 0.95 --max-unlabeled-per-epoch 2000 --resume

## 6. Apple-only classifier (4 classes)
Needed by the PlantDoc and Grad-CAM steps, which work on the 4 Apple classes.

In [ ]:
!python -m src.training.train_classifier --data-dir {DATA}/apple \
    --output-dir {RUN_DIR}/baseline_apple_only --mode full --epochs 15 --resume

## 7. U-Net leaf segmentation

In [ ]:
if not done(f'{RUN_DIR}/unet/seg_samples.png'):
    !python -m src.training.train_segmentation --images-dir {DATA}/seg/images \
        --masks-dir {DATA}/seg/masks --output-dir {RUN_DIR}/unet --epochs 15

## 8. DCGAN on the rarest Apple class (optional, ~20–30 min)

In [ ]:
if not done(f'{RUN_DIR}/gan_cedar_rust/real_vs_generated.png'):
    !python -m src.training.train_gan --data-dir {DATA}/apple/train/Apple___Cedar_apple_rust \
        --output-dir {RUN_DIR}/gan_cedar_rust --epochs 100

## 9. Cross-dataset test: PlantVillage → PlantDoc (real field photos)

In [ ]:
if not os.path.exists('/content/PlantDoc-Dataset'):
    !git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /content/PlantDoc-Dataset
if not done(f'{RUN_DIR}/cross_dataset_eval/cross_dataset_results.json'):
    !python -m src.eval.cross_dataset_eval --checkpoint {RUN_DIR}/baseline_apple_only/best_model.pt \
        --plantvillage-val-dir {DATA}/apple/val --plantdoc-dir /content/PlantDoc-Dataset/test \
        --output-dir {RUN_DIR}/cross_dataset_eval

## 10. Grad-CAM lesion localization (no box labels used)

In [ ]:
if not done(f'{RUN_DIR}/gradcam_boxes'):
    !python -m src.interpretability.pseudo_boxes --checkpoint {RUN_DIR}/baseline_apple_only/best_model.pt \
        --data-dir {DATA}/apple/val --output-dir {RUN_DIR}/gradcam_boxes --num-samples 16

## 11. Class-incremental few-shot learning
Runs naive fine-tuning and replay + distillation back to back.

In [ ]:
if not done(f'{RUN_DIR}/incremental/incremental_results.json'):
    !python -m src.incremental.train_incremental --data-dir {DATA}/apple \
        --output-dir {RUN_DIR}/incremental --k-shot 20 --epochs-per-task 15

## 12. Build the report
Writes `RESULTS.md`, all figures, and fills the results block in `README.md`. Safe to run after any subset of steps.

In [ ]:
!python -m src.utils.generate_report --run-dir {RUN_DIR} --output-dir {RUN_DIR}/report \
    --assets-dir {RUN_DIR}/report/assets --readme README.md
!cp README.md {RUN_DIR}/report/README.md
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{RUN_DIR}/report/assets/*.png'))[:6]:
    print(os.path.basename(f)); display(Image(f, width=520))

## 13. Download the report
`report.zip` holds the filled-in `README.md`, `RESULTS.md` and `assets/`. Copy those into your local repo before committing.

In [ ]:
!cd {RUN_DIR} && zip -q -r /content/report.zip report
from google.colab import files
files.download('/content/report.zip')